In [ ]:
# ============================================
# NOTEBOOK 2 : Traitement Distribué d'Images avec PySpark
# Dataset : Oxford-IIIT Pet Dataset (7,349 images)
# Objectifs : Chargement, Transformation, Feature Extraction, Partitionnement
# ============================================

# Installation de PySpark (pour Google Colab)
!pip install pyspark --break-system-packages -q

print("✓ PySpark installé")

# Imports
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
import os

# Créer la SparkSession
spark = SparkSession.builder \
    .appName("Oxford Pets - Traitement Images") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

print(f"✓ Spark {spark.version} démarré")
print(f"✓ Spark UI: http://localhost:4040")
print("="*70)

✓ PySpark installé
✓ Spark 4.0.2 démarré
✓ Spark UI: http://localhost:4040


In [ ]:
# Ouvrir SparkUI en suivant le lien
from google.colab import output
output.serve_kernel_port_as_window(4040, path='/jobs/index.html')

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [ ]:
# ============================================
# INSTALLATION DES DÉPENDANCES
# ============================================

print("\n📦 Installation des dépendances")
print("="*70)

!pip install Pillow --break-system-packages -q
!pip install requests --break-system-packages -q

print("✓ Pillow installé (traitement d'images)")
print("✓ Requests installé")


📦 Installation des dépendances
✓ Pillow installé (traitement d'images)
✓ Requests installé


In [ ]:
# ============================================
# IMPORTATION DU MONITEUR SPARK
# ============================================

print("\n🔧 Configuration du Job Monitor")
print("="*70)

# Code du SparkJobMonitor (version simplifiée pour Colab)
import uuid
from typing import Callable, Any, Dict

class SparkJobMonitor:
    """Moniteur simplifié pour Google Colab"""

    def __init__(self, spark):
        self.spark = spark
        self.tracker = spark.sparkContext.statusTracker()
        self.job_history = []

    def execute_and_monitor(self, func: Callable, name: str) -> Any:
        job_id = str(uuid.uuid4())[:8]
        self.spark.sparkContext.setJobGroup(job_id, name)

        print(f"\n{'='*70}")
        print(f"🔵 {name}")
        print(f"{'='*70}")
        print(f"📌 Tracking ID: {job_id}")

        start_time = time.time()

        try:
            result = func()
            elapsed = time.time() - start_time
            status = "✅ SUCCESS"
            error = None
        except Exception as e:
            elapsed = time.time() - start_time
            status = "❌ FAILED"
            error = str(e)
            result = None
            print(f"\n❌ Erreur: {error}")

        spark_job_ids = list(self.tracker.getJobIdsForGroup(job_id))

        job_record = {
            'name': name,
            'tracking_id': job_id,
            'spark_job_ids': spark_job_ids,
            'duration': elapsed,
            'status': status,
            'error': error,
        }
        self.job_history.append(job_record)

        print(f"\n{status}")
        print(f"⏱️  Durée: {elapsed:.2f}s")
        print(f"📊 Spark Job ID(s): {spark_job_ids}")

        self.spark.sparkContext.setLocalProperty("spark.jobGroup.id", None)

        return result

    def show_history(self) -> None:
        print("\n" + "="*70)
        print("📜 HISTORIQUE DES JOBS")
        print("="*70)

        if not self.job_history:
            print("\n⚠️  Aucun job enregistré")
            return

        total_duration = sum(job['duration'] for job in self.job_history)
        success_count = sum(1 for job in self.job_history if job['status'] == "✅ SUCCESS")

        print(f"\n📊 Statistiques globales:")
        print(f"   Total jobs: {len(self.job_history)}")
        print(f"   Réussis: {success_count}")
        print(f"   Échoués: {len(self.job_history) - success_count}")
        print(f"   Durée totale: {total_duration:.2f}s")
        print(f"\n{'='*70}")

        for i, job in enumerate(self.job_history, 1):
            print(f"\n{i}. {job['name']}")
            print(f"   Tracking ID: {job['tracking_id']}")
            print(f"   Durée: {job['duration']:.2f}s")
            print(f"   Status: {job['status']}")

# Initialiser le moniteur
monitor = SparkJobMonitor(spark)

print("✓ SparkJobMonitor initialisé")


🔧 Configuration du Job Monitor
✓ SparkJobMonitor initialisé


In [ ]:
# ============================================
# PARTIE 1 : Téléchargement du Dataset Oxford Pets
# ============================================

print("\n📥 PARTIE 1 : Téléchargement du Dataset")
print("="*70)

import requests
import tarfile

# Créer les dossiers
data_dir = "/content/oxford_pets"
os.makedirs(data_dir, exist_ok=True)

# URLs
images_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz"
annotations_url = "https://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz"

images_tar = os.path.join(data_dir, "images.tar.gz")
annotations_tar = os.path.join(data_dir, "annotations.tar.gz")

# Télécharger les images
if not os.path.exists(os.path.join(data_dir, "images")):
    print(f"\n⏳ Téléchargement des images (~800 MB)...")
    print("   Cela peut prendre 5-10 minutes selon votre connexion...")

    response = requests.get(images_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))

    with open(images_tar, 'wb') as f:
        downloaded = 0
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if downloaded % (50 * 1024 * 1024) == 0:
                    print(f"   📊 Téléchargé : {downloaded / (1024**2):.0f} / {total_size / (1024**2):.0f} MB")

    print(f"\n✓ Images téléchargées")

    # Extraction
    print("⏳ Extraction de l'archive...")
    with tarfile.open(images_tar, 'r:gz') as tar:
        tar.extractall(data_dir)

    print("✓ Images extraites")

    # Nettoyage
    os.remove(images_tar)
    print("✓ Archive supprimée")
else:
    print("✓ Images déjà présentes")

# Télécharger les annotations
if not os.path.exists(os.path.join(data_dir, "annotations")):
    print(f"\n⏳ Téléchargement des annotations...")

    response = requests.get(annotations_url, stream=True)
    with open(annotations_tar, 'wb') as f:
        f.write(response.content)

    print("✓ Annotations téléchargées")

    # Extraction
    print("⏳ Extraction...")
    with tarfile.open(annotations_tar, 'r:gz') as tar:
        tar.extractall(data_dir)

    print("✓ Annotations extraites")
    os.remove(annotations_tar)
else:
    print("✓ Annotations déjà présentes")

# Afficher la structure
images_path = os.path.join(data_dir, "images")
nb_images = len([f for f in os.listdir(images_path) if f.endswith('.jpg')])

print(f"\n📊 Dataset téléchargé :")
print(f"   📁 Chemin : {data_dir}")
print(f"   🖼️  Nombre d'images : {nb_images:,}")

# Exemples de fichiers
print(f"\n📝 Exemples de fichiers :")
for i, filename in enumerate(sorted(os.listdir(images_path))[:5], 1):
    print(f"   {i}. {filename}")


📥 PARTIE 1 : Téléchargement du Dataset

⏳ Téléchargement des images (~800 MB)...
   Cela peut prendre 5-10 minutes selon votre connexion...
   📊 Téléchargé : 50 / 755 MB
   📊 Téléchargé : 100 / 755 MB
   📊 Téléchargé : 150 / 755 MB
   📊 Téléchargé : 200 / 755 MB
   📊 Téléchargé : 250 / 755 MB
   📊 Téléchargé : 300 / 755 MB
   📊 Téléchargé : 350 / 755 MB
   📊 Téléchargé : 400 / 755 MB
   📊 Téléchargé : 450 / 755 MB
   📊 Téléchargé : 500 / 755 MB
   📊 Téléchargé : 550 / 755 MB
   📊 Téléchargé : 600 / 755 MB
   📊 Téléchargé : 650 / 755 MB
   📊 Téléchargé : 700 / 755 MB
   📊 Téléchargé : 750 / 755 MB

✓ Images téléchargées
⏳ Extraction de l'archive...


/tmp/ipython-input-4019948515.py:44: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(data_dir)


✓ Images extraites
✓ Archive supprimée

⏳ Téléchargement des annotations...
✓ Annotations téléchargées
⏳ Extraction...


/tmp/ipython-input-4019948515.py:67: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(data_dir)


✓ Annotations extraites

📊 Dataset téléchargé :
   📁 Chemin : /content/oxford_pets
   🖼️  Nombre d'images : 7,390

📝 Exemples de fichiers :
   1. Abyssinian_1.jpg
   2. Abyssinian_10.jpg
   3. Abyssinian_100.jpg
   4. Abyssinian_100.mat
   5. Abyssinian_101.jpg


In [ ]:
# ============================================
# PARTIE 2 : Chargement Distribué avec BinaryFile
# ============================================

print("\n📂 PARTIE 2 : Chargement Distribué des Images")
print("="*70)

print("""
🧠 CONCEPT : spark.read.format("binaryFile")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Lit les fichiers binaires (images, PDF, etc.)
• Charge en PARALLÈLE sur tous les workers
• LAZY : ne lit pas vraiment les données encore
• Colonnes automatiques : path, modificationTime, length, content

Schéma :
  ├─ path : string (chemin du fichier)
  ├─ modificationTime : timestamp
  ├─ length : long (taille en bytes)
  └─ content : binary (contenu brut de l'image)
""")

print("\n⏱️ Chargement des 7,000+ images...")
start = time.time()

# Chargement LAZY
df_images = spark.read.format("binaryFile") \
    .option("pathGlobFilter", "*.jpg") \
    .load(images_path)

elapsed = time.time() - start

print(f"✓ 'Chargement' effectué en {elapsed:.4f}s")

print("\n💡 OBSERVATION :")
print("   • Temps quasi-instantané (< 1 seconde)")
print("   • AUCUNE image lue en mémoire")
print("   • Spark a juste scanné le répertoire")
print("   • Les métadonnées sont disponibles")

print("\n📋 Schéma du DataFrame :")
df_images.printSchema()

print("\n🔍 Aucun job dans Spark UI pour l'instant (LAZY !)")


📂 PARTIE 2 : Chargement Distribué des Images

🧠 CONCEPT : spark.read.format("binaryFile")
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Lit les fichiers binaires (images, PDF, etc.)
• Charge en PARALLÈLE sur tous les workers
• LAZY : ne lit pas vraiment les données encore
• Colonnes automatiques : path, modificationTime, length, content

Schéma :
  ├─ path : string (chemin du fichier)
  ├─ modificationTime : timestamp
  ├─ length : long (taille en bytes)
  └─ content : binary (contenu brut de l'image)


⏱️ Chargement des 7,000+ images...
✓ 'Chargement' effectué en 2.8417s

💡 OBSERVATION :
   • Temps quasi-instantané (< 1 seconde)
   • AUCUNE image lue en mémoire
   • Spark a juste scanné le répertoire
   • Les métadonnées sont disponibles

📋 Schéma du DataFrame :
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- content: binary (nullable = true)


🔍 Aucun job dans Spark UI pour l'instant

In [ ]:
# ============================================
# PARTIE 3 : Première ACTION - Exploration
# ============================================

print("\n⚡ PARTIE 3 : Première ACTION - count()")
print("="*70)

print("\n💥 Maintenant on déclenche une ACTION : count()")
print("   Cela va LIRE les métadonnées des fichiers")

result = monitor.execute_and_monitor(
    lambda: df_images.count(),
    "JOB 1: Compter les images"
)

print(f"\n✓ {result:,} images chargées")

print("\n📊 Affichage d'un échantillon (ACTION : show())")

result = monitor.execute_and_monitor(
    lambda: df_images.select("path", "length", "modificationTime").show(5, truncate=False),
    "JOB 2: Afficher un échantillon"
)


⚡ PARTIE 3 : Première ACTION - count()

💥 Maintenant on déclenche une ACTION : count()
   Cela va LIRE les métadonnées des fichiers

🔵 JOB 1: Compter les images
📌 Tracking ID: c3403fc4

✅ SUCCESS
⏱️  Durée: 8.91s
📊 Spark Job ID(s): [1, 0]

✓ 7,390 images chargées

📊 Affichage d'un échantillon (ACTION : show())

🔵 JOB 2: Afficher un échantillon
📌 Tracking ID: 1c58512a
+-----------------------------------------------------+-------+-------------------+
|path                                                 |length |modificationTime   |
+-----------------------------------------------------+-------+-------------------+
|file:/content/oxford_pets/images/Egyptian_Mau_14.jpg |1125149|2012-06-18 16:52:58|
|file:/content/oxford_pets/images/Egyptian_Mau_165.jpg|1061923|2012-06-18 16:53:21|
|file:/content/oxford_pets/images/Egyptian_Mau_162.jpg|968009 |2012-06-18 16:54:01|
|file:/content/oxford_pets/images/Egyptian_Mau_196.jpg|661645 |2012-06-18 16:54:12|
|file:/content/oxford_pets/images/Abyssin

In [ ]:
# ============================================
# PARTIE 4 : Extraction des Métadonnées
# ============================================

print("\n🏷️ PARTIE 4 : Extraction des Métadonnées (TRANSFORMATIONS)")
print("="*70)

print("\n🔄 On va extraire des informations depuis les noms de fichiers")
print("   Format : 'Abyssinian_1.jpg' → Race: Abyssinian, ID: 1")

print("\n⏱️ Application des transformations (LAZY)...")
start = time.time()

# Extraire le nom de fichier
df_with_meta = df_images.withColumn(
    "filename",
    F.element_at(F.split(F.col("path"), "/"), -1)
)

# Extraire la race (avant le dernier underscore)
df_with_meta = df_with_meta.withColumn(
    "race_raw",
    F.regexp_replace(F.col("filename"), r"_\d+\.jpg$", "")
)

# Normaliser le nom de la race
df_with_meta = df_with_meta.withColumn(
    "race",
    F.regexp_replace(F.col("race_raw"), "_", " ")
)

# Extraire l'ID
df_with_meta = df_with_meta.withColumn(
    "image_id",
    F.regexp_extract(F.col("filename"), r"_(\d+)\.jpg$", 1).cast("int")
)

# Catégorie (chat ou chien)
# Les chats ont une majuscule au début, les chiens sont en minuscules après la 1ère lettre
df_with_meta = df_with_meta.withColumn(
    "species",
    F.when(F.col("race_raw").rlike("^[A-Z][a-z]+_"), "Cat")
     .otherwise("Dog")
)

# Taille en MB
df_with_meta = df_with_meta.withColumn(
    "size_mb",
    (F.col("length") / (1024 * 1024)).cast("decimal(10,2)")
)

elapsed = time.time() - start

print(f"✓ Transformations ajoutées en {elapsed:.4f}s")

print("\n💡 OBSERVATION :")
print("   • Très rapide (< 0.01s)")
print("   • RIEN n'a été calculé (LAZY)")
print("   • Spark a juste enrichi le plan d'exécution")

print("\n⚡ Maintenant on affiche (ACTION) :")

result = monitor.execute_and_monitor(
    lambda: df_with_meta.select("filename", "race", "species", "image_id", "size_mb")
        .show(10, truncate=False),
    "JOB 3: Afficher métadonnées enrichies"
)


🏷️ PARTIE 4 : Extraction des Métadonnées (TRANSFORMATIONS)

🔄 On va extraire des informations depuis les noms de fichiers
   Format : 'Abyssinian_1.jpg' → Race: Abyssinian, ID: 1

⏱️ Application des transformations (LAZY)...
✓ Transformations ajoutées en 0.3129s

💡 OBSERVATION :
   • Très rapide (< 0.01s)
   • RIEN n'a été calculé (LAZY)
   • Spark a juste enrichi le plan d'exécution

⚡ Maintenant on affiche (ACTION) :

🔵 JOB 3: Afficher métadonnées enrichies
📌 Tracking ID: 1011f39b
+--------------------------+------------------+-------+--------+-------+
|filename                  |race              |species|image_id|size_mb|
+--------------------------+------------------+-------+--------+-------+
|Egyptian_Mau_14.jpg       |Egyptian Mau      |Cat    |14      |1.07   |
|Egyptian_Mau_165.jpg      |Egyptian Mau      |Cat    |165     |1.01   |
|Egyptian_Mau_162.jpg      |Egyptian Mau      |Cat    |162     |0.92   |
|Egyptian_Mau_196.jpg      |Egyptian Mau      |Cat    |196     |0.63   |


In [ ]:
# ============================================
# PARTIE 5 : Statistiques sur les Métadonnées
# ============================================

print("\n📊 PARTIE 5 : Statistiques sur le Dataset")
print("="*70)

print("\n1️⃣ Distribution des espèces (Chats vs Chiens)")

result = monitor.execute_and_monitor(
    lambda: df_with_meta.groupBy("species")
        .agg(
            F.count("*").alias("nb_images"),
            F.avg("size_mb").alias("taille_moyenne_mb"),
            F.min("size_mb").alias("taille_min_mb"),
            F.max("size_mb").alias("taille_max_mb")
        )
        .show(truncate=False),
    "JOB 4: Stats par espèce"
)

print("\n2️⃣ Top 10 races avec le plus d'images")

result = monitor.execute_and_monitor(
    lambda: df_with_meta.groupBy("race", "species")
        .agg(F.count("*").alias("nb_images"))
        .orderBy(F.col("nb_images").desc())
        .limit(10)
        .show(truncate=False),
    "JOB 5: Top 10 races"
)

print("\n3️⃣ Distribution des tailles de fichiers")

result = monitor.execute_and_monitor(
    lambda: df_with_meta.select("size_mb").describe().show(),
    "JOB 6: Statistiques des tailles"
)

print("\n4️⃣ Nombre de races par espèce")

result = monitor.execute_and_monitor(
    lambda: df_with_meta.groupBy("species")
        .agg(F.countDistinct("race").alias("nb_races"))
        .show(),
    "JOB 7: Nombre de races"
)


📊 PARTIE 5 : Statistiques sur le Dataset

1️⃣ Distribution des espèces (Chats vs Chiens)

🔵 JOB 4: Stats par espèce
📌 Tracking ID: bfd0fb09
+-------+---------+-----------------+-------------+-------------+
|species|nb_images|taille_moyenne_mb|taille_min_mb|taille_max_mb|
+-------+---------+-----------------+-------------+-------------+
|Cat    |800      |0.090700         |0.00         |1.07         |
|Dog    |6590     |0.103704         |0.00         |0.59         |
+-------+---------+-----------------+-------------+-------------+


✅ SUCCESS
⏱️  Durée: 8.81s
📊 Spark Job ID(s): [5, 4]

2️⃣ Top 10 races avec le plus d'images

🔵 JOB 5: Top 10 races
📌 Tracking ID: bcca4701
+----------------------+-------+---------+
|race                  |species|nb_images|
+----------------------+-------+---------+
|Bombay                |Dog    |200      |
|german shorthaired    |Dog    |200      |
|english cocker spaniel|Dog    |200      |
|Bengal                |Dog    |200      |
|keeshond           

In [ ]:
# ============================================
# PARTIE 6 : Décodage des Images (UDF)
# ============================================

print("\n🖼️ PARTIE 6 : Décodage et Analyse des Images")
print("="*70)

print("""
🧠 CONCEPT : UDF (User Defined Function)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Permet d'appliquer une fonction Python personnalisée
• Fonctionne sur chaque ligne du DataFrame
• Exécutée en PARALLÈLE sur tous les workers
• Utile pour des traitements complexes (décoder images, OCR, etc.)

Ici : Décoder les images binaires pour extraire dimensions
""")

from pyspark.sql.functions import udf
from PIL import Image
import io

# UDF pour extraire les dimensions
@udf(returnType=StructType([
    StructField("width", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("channels", IntegerType(), True),
    StructField("format", StringType(), True)
]))
def extract_image_info(image_bytes):
    """Extrait les dimensions d'une image depuis son contenu binaire"""
    try:
        img = Image.open(io.BytesIO(image_bytes))

        # Nombre de channels
        if img.mode == 'RGB':
            channels = 3
        elif img.mode == 'RGBA':
            channels = 4
        elif img.mode == 'L':
            channels = 1
        else:
            channels = len(img.getbands())

        return {
            'width': img.width,
            'height': img.height,
            'channels': channels,
            'format': img.format
        }
    except Exception as e:
        return None

print("\n⏱️ Application de l'UDF (TRANSFORMATION - lazy)...")
start = time.time()

df_with_dims = df_with_meta.withColumn(
    "image_info",
    extract_image_info(F.col("content"))
)

# Extraire les champs
df_with_dims = df_with_dims \
    .withColumn("width", F.col("image_info.width")) \
    .withColumn("height", F.col("image_info.height")) \
    .withColumn("channels", F.col("image_info.channels")) \
    .withColumn("format", F.col("image_info.format")) \
    .withColumn("pixels", F.col("width") * F.col("height")) \
    .withColumn("aspect_ratio", (F.col("width") / F.col("height")).cast("decimal(10,2)")) \
    .drop("image_info")

elapsed = time.time() - start

print(f"✓ UDF ajoutée au plan en {elapsed:.4f}s (LAZY)")

print("\n⚡ Maintenant on EXÉCUTE l'UDF sur toutes les images (ACTION) :")
print("   ⚠️  Cela va prendre du temps (décodage de 7000+ images)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.select("filename", "race", "width", "height", "channels", "format", "aspect_ratio")
        .show(10, truncate=False),
    "JOB 8: Décoder les images et extraire dimensions"
)

print("\n💡 OBSERVATIONS :")
print("   • Le job a pris plus de temps (décodage réel)")
print("   • L'UDF a été exécutée EN PARALLÈLE sur tous les workers")
print("   • Chaque worker a décodé une partie des images")


🖼️ PARTIE 6 : Décodage et Analyse des Images

🧠 CONCEPT : UDF (User Defined Function)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Permet d'appliquer une fonction Python personnalisée
• Fonctionne sur chaque ligne du DataFrame
• Exécutée en PARALLÈLE sur tous les workers
• Utile pour des traitements complexes (décoder images, OCR, etc.)

Ici : Décoder les images binaires pour extraire dimensions


⏱️ Application de l'UDF (TRANSFORMATION - lazy)...
✓ UDF ajoutée au plan en 0.5160s (LAZY)

⚡ Maintenant on EXÉCUTE l'UDF sur toutes les images (ACTION) :
   ⚠️  Cela va prendre du temps (décodage de 7000+ images)

🔵 JOB 8: Décoder les images et extraire dimensions
📌 Tracking ID: 3e6106f1
+--------------------------+------------------+-----+------+--------+------+------------+
|filename                  |race              |width|height|channels|format|aspect_ratio|
+--------------------------+------------------+-----+------+--------+------+------------+
|Egyptian_Mau_14.jpg 

In [ ]:
# ============================================
# PARTIE 7 : Statistiques sur les Dimensions
# ============================================

print("\n📐 PARTIE 7 : Analyse des Dimensions")
print("="*70)

print("\n1️⃣ Distribution des dimensions")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.select("width", "height", "channels").describe().show(),
    "JOB 9: Stats dimensions"
)

print("\n2️⃣ Formats d'images")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("format")
        .agg(F.count("*").alias("nb_images"))
        .show(),
    "JOB 10: Distribution des formats"
)

print("\n3️⃣ Images carrées vs rectangulaires")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.withColumn(
        "shape_type",
        F.when(F.col("width") == F.col("height"), "Carré")
         .when(F.col("width") > F.col("height"), "Paysage")
         .otherwise("Portrait")
    ).groupBy("shape_type")
     .agg(F.count("*").alias("nb_images"))
     .show(),
    "JOB 11: Distribution des orientations"
)

print("\n4️⃣ Tailles les plus courantes")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("width", "height")
        .agg(F.count("*").alias("nb_images"))
        .orderBy(F.col("nb_images").desc())
        .limit(10)
        .show(),
    "JOB 12: Top 10 tailles"
)


📐 PARTIE 7 : Analyse des Dimensions

1️⃣ Distribution des dimensions

🔵 JOB 9: Stats dimensions
📌 Tracking ID: 5e288040
+-------+------------------+------------------+-------------------+
|summary|             width|            height|           channels|
+-------+------------------+------------------+-------------------+
|  count|              7390|              7390|               7390|
|   mean| 436.7451962110961| 390.9136671177267| 2.9979702300405955|
| stddev|115.87730011126129|109.58632393092991|0.07262228029252112|
|    min|               114|               103|                  1|
|    max|              3264|              2606|                  4|
+-------+------------------+------------------+-------------------+


✅ SUCCESS
⏱️  Durée: 63.21s
📊 Spark Job ID(s): [15, 14]

2️⃣ Formats d'images

🔵 JOB 10: Distribution des formats
📌 Tracking ID: e3afeba7
+------+---------+
|format|nb_images|
+------+---------+
|  JPEG|     7380|
|   PNG|        4|
|   GIF|        6|
+------+-----

In [ ]:
# ============================================
# PARTIE 8 : Cache et Performance
# ============================================

print("\n🚀 PARTIE 8 : Impact du Cache")
print("="*70)

print("""
🧠 CONCEPT : Cache
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• cache() ou persist() garde les données en mémoire
• Évite de recalculer les transformations coûteuses
• Utile quand on réutilise le même DataFrame
• Libérer avec unpersist() quand terminé
""")

print("\n1️⃣ Première exécution SANS cache")

result1 = monitor.execute_and_monitor(
    lambda: df_with_dims.filter(F.col("species") == "Cat").count(),
    "JOB 13: Count chats (sans cache)"
)

print(f"✓ {result1:,} images de chats")

print("\n2️⃣ Mise en cache du DataFrame")
df_with_dims.cache()

print("\n3️⃣ Deuxième exécution AVEC cache (premier passage - mise en cache)")

result2 = monitor.execute_and_monitor(
    lambda: df_with_dims.filter(F.col("species") == "Cat").count(),
    "JOB 14: Count chats (mise en cache)"
)

print("\n4️⃣ Troisième exécution DEPUIS le cache")

result3 = monitor.execute_and_monitor(
    lambda: df_with_dims.filter(F.col("species") == "Cat").count(),
    "JOB 15: Count chats (depuis cache)"
)

print("\n💡 COMPARAISON :")
print("   • JOB 13 : Lecture + décodage depuis fichiers")
print("   • JOB 14 : Lecture + décodage + mise en cache")
print("   • JOB 15 : Lecture DEPUIS cache (beaucoup plus rapide !)")

print("\n🌐 Dans Spark UI → Storage :")
print("   • Vous voyez le DataFrame en cache")
print("   • Taille mémoire utilisée")


🚀 PARTIE 8 : Impact du Cache

🧠 CONCEPT : Cache
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• cache() ou persist() garde les données en mémoire
• Évite de recalculer les transformations coûteuses
• Utile quand on réutilise le même DataFrame
• Libérer avec unpersist() quand terminé


1️⃣ Première exécution SANS cache

🔵 JOB 13: Count chats (sans cache)
📌 Tracking ID: 690e8a3c

✅ SUCCESS
⏱️  Durée: 1.91s
📊 Spark Job ID(s): [23, 22]
✓ 800 images de chats

2️⃣ Mise en cache du DataFrame

3️⃣ Deuxième exécution AVEC cache (premier passage - mise en cache)

🔵 JOB 14: Count chats (mise en cache)
📌 Tracking ID: 93f7e361

✅ SUCCESS
⏱️  Durée: 64.15s
📊 Spark Job ID(s): [26, 25, 24]

4️⃣ Troisième exécution DEPUIS le cache

🔵 JOB 15: Count chats (depuis cache)
📌 Tracking ID: ee7d5397

✅ SUCCESS
⏱️  Durée: 2.09s
📊 Spark Job ID(s): [28, 27]

💡 COMPARAISON :
   • JOB 13 : Lecture + décodage depuis fichiers
   • JOB 14 : Lecture + décodage + mise en cache
   • JOB 15 : Lecture DEPUIS

In [ ]:
# ============================================
# PARTIE 9 : Redimensionnement Distribué
# ============================================

print("\n✂️ PARTIE 9 : Redimensionnement Massif d'Images")
print("="*70)

print("""
🧠 CONCEPT : Map Operation Distribuée
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Redimensionner 7000+ images en parallèle
• Chaque worker traite une partie des images
• UDF appliquée de manière distribuée
• Gain de temps significatif vs traitement séquentiel
""")

# UDF pour redimensionner
@udf(returnType=BinaryType())
def resize_image(image_bytes, target_width=224, target_height=224):
    """Redimensionne une image à une taille fixe"""
    try:
        img = Image.open(io.BytesIO(image_bytes))

        # Redimensionner
        img_resized = img.resize((target_width, target_height), Image.LANCZOS)

        # Convertir en bytes
        buffer = io.BytesIO()
        img_resized.save(buffer, format='JPEG')

        return buffer.getvalue()
    except Exception as e:
        return None

print("\n⏱️ Redimensionnement de TOUTES les images à 224×224...")
print("   (Taille standard pour les modèles de deep learning)")

# On va travailler sur un échantillon pour aller plus vite
df_sample = df_with_dims.limit(1000)

result = monitor.execute_and_monitor(
    lambda: df_sample.withColumn(
        "resized_content",
        resize_image(F.col("content"), F.lit(224), F.lit(224))
    ).select("filename", "width", "height")
     .show(10),
    "JOB 16: Redimensionner 1000 images"
)

print("\n💡 OBSERVATIONS :")
print("   • Redimensionnement distribué sur tous les workers")
print("   • Chaque worker traite sa partition en parallèle")
print("   • Gain de temps vs traitement séquentiel")


✂️ PARTIE 9 : Redimensionnement Massif d'Images

🧠 CONCEPT : Map Operation Distribuée
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Redimensionner 7000+ images en parallèle
• Chaque worker traite une partie des images
• UDF appliquée de manière distribuée
• Gain de temps significatif vs traitement séquentiel


⏱️ Redimensionnement de TOUTES les images à 224×224...
   (Taille standard pour les modèles de deep learning)

🔵 JOB 16: Redimensionner 1000 images
📌 Tracking ID: fb4216fd
+--------------------+-----+------+
|            filename|width|height|
+--------------------+-----+------+
| Egyptian_Mau_14.jpg|  582|   800|
|Egyptian_Mau_165.jpg| 2560|  1920|
|Egyptian_Mau_162.jpg| 3264|  2448|
|Egyptian_Mau_196.jpg| 1886|  2606|
|  Abyssinian_178.jpg|  400|   400|
|       Bombay_29.jpg| 1600|  1200|
|Egyptian_Mau_182.jpg| 1999|  1499|
|  Abyssinian_170.jpg| 1600|  1200|
|   Abyssinian_40.jpg| 1646|  1278|
|german_shorthaire...|  500|   500|
+--------------------+-----+---

In [ ]:
# ============================================
# PARTIE 10 : Feature Extraction Simple
# ============================================

print("\n🎨 PARTIE 10 : Extraction de Features Simples")
print("="*70)

print("""
🧠 CONCEPT : Feature Engineering Distribué
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Extraire des caractéristiques de chaque image
• Couleur dominante (moyenne RGB)
• Luminosité moyenne
• Tous les calculs en parallèle
""")

# UDF pour extraire les features
@udf(returnType=StructType([
    StructField("avg_red", FloatType(), True),
    StructField("avg_green", FloatType(), True),
    StructField("avg_blue", FloatType(), True),
    StructField("brightness", FloatType(), True)
]))
def extract_color_features(image_bytes):
    """Extrait les features de couleur"""
    try:
        img = Image.open(io.BytesIO(image_bytes))

        # Convertir en RGB si nécessaire
        if img.mode != 'RGB':
            img = img.convert('RGB')

        # Réduire la taille pour accélérer
        img_small = img.resize((50, 50))

        # Extraire les pixels
        pixels = list(img_small.getdata())

        # Calculer les moyennes
        avg_r = sum(p[0] for p in pixels) / len(pixels)
        avg_g = sum(p[1] for p in pixels) / len(pixels)
        avg_b = sum(p[2] for p in pixels) / len(pixels)

        # Luminosité (formule standard)
        brightness = (0.299 * avg_r + 0.587 * avg_g + 0.114 * avg_b)

        return {
            'avg_red': float(avg_r),
            'avg_green': float(avg_g),
            'avg_blue': float(avg_b),
            'brightness': float(brightness)
        }
    except Exception as e:
        return None

print("\n⏱️ Extraction des features de couleur...")

result = monitor.execute_and_monitor(
    lambda: df_sample.withColumn("color_features", extract_color_features(F.col("content")))
        .withColumn("avg_red", F.col("color_features.avg_red"))
        .withColumn("avg_green", F.col("color_features.avg_green"))
        .withColumn("avg_blue", F.col("color_features.avg_blue"))
        .withColumn("brightness", F.col("color_features.brightness"))
        .select("filename", "race", "avg_red", "avg_green", "avg_blue", "brightness")
        .show(10),
    "JOB 17: Extraction features couleur"
)

print("\n💡 Utilisations possibles :")
print("   • Filtrer les images trop sombres/claires")
print("   • Grouper par couleur dominante")
print("   • Détecter les images en noir et blanc")


🎨 PARTIE 10 : Extraction de Features Simples

🧠 CONCEPT : Feature Engineering Distribué
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Extraire des caractéristiques de chaque image
• Couleur dominante (moyenne RGB)
• Luminosité moyenne
• Tous les calculs en parallèle


⏱️ Extraction des features de couleur...

🔵 JOB 17: Extraction features couleur
📌 Tracking ID: 93a3d89a
+--------------------+------------------+-------+---------+--------+----------+
|            filename|              race|avg_red|avg_green|avg_blue|brightness|
+--------------------+------------------+-------+---------+--------+----------+
| Egyptian_Mau_14.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_165.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_162.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|Egyptian_Mau_196.jpg|      Egyptian Mau|   NULL|     NULL|    NULL|      NULL|
|  Abyssinian_178.jpg|        Abyssinian|   NULL| 

In [ ]:
# ============================================
# PARTIE 11 : Partitionnement et Sauvegarde
# ============================================

print("\n💾 PARTIE 11 : Partitionnement et Sauvegarde")
print("="*70)

print("""
🧠 CONCEPT : Partitionnement par Catégorie
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Organiser les images par espèce (Cat/Dog)
• Puis par race
• Permet la lecture sélective (partition pruning)
• Format Parquet pour stocker les données binaires
""")

output_path = "/content/oxford_pets_processed"

print("\n📝 Sauvegarde partitionnée par espèce et race...")
print("   (Sur un échantillon de 1000 images)")

# Préparer le DataFrame
df_to_save = df_sample.select(
    "filename",
    "species",
    "race",
    "width",
    "height",
    "channels",
    "size_mb",
    "content"
)

result = monitor.execute_and_monitor(
    lambda: df_to_save.write
        .mode("overwrite")
        .partitionBy("species", "race")
        .parquet(output_path),
    "JOB 18: Sauvegarde partitionnée"
)

print(f"\n✓ Données sauvegardées dans: {output_path}")

# Afficher la structure
print("\n📁 Structure des partitions:")
for root, dirs, files in os.walk(output_path):
    level = root.replace(output_path, '').count(os.sep)
    indent = '   ' * level
    folder = os.path.basename(root)
    if folder and level < 3:
        print(f"{indent}📂 {folder}/")

    if files and level < 3:
        parquet_files = [f for f in files if f.endswith('.parquet')]
        if parquet_files:
            print(f"{indent}   📄 {len(parquet_files)} fichier(s) .parquet")


💾 PARTIE 11 : Partitionnement et Sauvegarde

🧠 CONCEPT : Partitionnement par Catégorie
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
• Organiser les images par espèce (Cat/Dog)
• Puis par race
• Permet la lecture sélective (partition pruning)
• Format Parquet pour stocker les données binaires


📝 Sauvegarde partitionnée par espèce et race...
   (Sur un échantillon de 1000 images)

🔵 JOB 18: Sauvegarde partitionnée
📌 Tracking ID: f45b96d7

✅ SUCCESS
⏱️  Durée: 30.57s
📊 Spark Job ID(s): [33, 32]

✓ Données sauvegardées dans: /content/oxford_pets_processed

📁 Structure des partitions:
📂 oxford_pets_processed/
   📂 species=Cat/
      📂 race=British Shorthair/
         📄 1 fichier(s) .parquet
      📂 race=Egyptian Mau/
         📄 1 fichier(s) .parquet
      📂 race=Russian Blue/
         📄 1 fichier(s) .parquet
      📂 race=Maine Coon/
         📄 1 fichier(s) .parquet
   📂 species=Dog/
      📂 race=wheaten terrier/
         📄 1 fichier(s) .parquet
      📂 race=shiba inu/
     

In [ ]:
# ============================================
# PARTIE 12 : Relecture et Partition Pruning
# ============================================

print("\n🔍 PARTIE 12 : Partition Pruning en Action")
print("="*70)

print("\n1️⃣ Lecture de TOUTES les partitions")

result1 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path).count(),
    "JOB 19: Lecture complète"
)

print(f"✓ {result1:,} images lues")

print("\n2️⃣ Lecture d'UNE SEULE espèce (partition pruning)")

result2 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter(F.col("species") == "Cat")
        .count(),
    "JOB 20: Lecture chats uniquement"
)

print(f"✓ {result2:,} images de chats")
print("💡 Spark a lu SEULEMENT le dossier species=Cat/")

print("\n3️⃣ Lecture d'UNE race spécifique")

result3 = monitor.execute_and_monitor(
    lambda: spark.read.parquet(output_path)
        .filter((F.col("species") == "Cat") & (F.col("race") == "Abyssinian"))
        .select("filename", "width", "height")
        .show(5),
    "JOB 21: Lecture Abyssinian uniquement"
)

print("💡 Spark a lu SEULEMENT species=Cat/race=Abyssinian/")

# Afficher le plan d'exécution
print("\n📋 Plan d'exécution (vérifier le partition pruning):")
spark.read.parquet(output_path) \
    .filter(F.col("species") == "Cat") \
    .explain()

print("\n🔍 Cherchez 'PartitionFilters: [isnotnull(species), (species = Cat)]'")


🔍 PARTIE 12 : Partition Pruning en Action

1️⃣ Lecture de TOUTES les partitions

🔵 JOB 19: Lecture complète
📌 Tracking ID: 9ca80ee4

✅ SUCCESS
⏱️  Durée: 1.84s
📊 Spark Job ID(s): [37, 36, 35, 34]
✓ 1,000 images lues

2️⃣ Lecture d'UNE SEULE espèce (partition pruning)

🔵 JOB 20: Lecture chats uniquement
📌 Tracking ID: b76f0af1

✅ SUCCESS
⏱️  Durée: 1.26s
📊 Spark Job ID(s): [41, 40, 39, 38]
✓ 66 images de chats
💡 Spark a lu SEULEMENT le dossier species=Cat/

3️⃣ Lecture d'UNE race spécifique

🔵 JOB 21: Lecture Abyssinian uniquement
📌 Tracking ID: a0a8dfe1
+--------+-----+------+
|filename|width|height|
+--------+-----+------+
+--------+-----+------+


✅ SUCCESS
⏱️  Durée: 0.95s
📊 Spark Job ID(s): [43, 42]
💡 Spark a lu SEULEMENT species=Cat/race=Abyssinian/

📋 Plan d'exécution (vérifier le partition pruning):
== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [filename#3115,width#3116,height#3117,channels#3118,size_mb#3119,content#3120,species#3121,race#3122] Batched: true, DataF

In [ ]:
# ============================================
# PARTIE 13 : Filtrage et Sélection Avancée
# ============================================

print("\n🔎 PARTIE 13 : Filtrage et Sélection")
print("="*70)

print("\n1️⃣ Images avec résolution minimale (> 300x300)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.filter((F.col("width") >= 300) & (F.col("height") >= 300))
        .groupBy("species")
        .agg(F.count("*").alias("nb_images"))
        .show(),
    "JOB 22: Filtrer résolution minimale"
)

print("\n2️⃣ Images presque carrées (aspect ratio entre 0.9 et 1.1)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.filter((F.col("aspect_ratio") >= 0.9) & (F.col("aspect_ratio") <= 1.1))
        .select("filename", "width", "height", "aspect_ratio")
        .show(10),
    "JOB 23: Images carrées"
)

print("\n3️⃣ Échantillonnage aléatoire stratifié (10 images par race)")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("race")
        .agg(F.collect_list("filename").alias("filenames"))
        .withColumn("sample", F.slice(F.col("filenames"), 1, 10))
        .select("race", F.size("sample").alias("nb_echantillon"))
        .show(10),
    "JOB 24: Échantillonnage par race"
)


🔎 PARTIE 13 : Filtrage et Sélection

1️⃣ Images avec résolution minimale (> 300x300)

🔵 JOB 22: Filtrer résolution minimale
📌 Tracking ID: 9ee62a54
+-------+---------+
|species|nb_images|
+-------+---------+
|    Cat|      632|
|    Dog|     5688|
+-------+---------+


✅ SUCCESS
⏱️  Durée: 7.67s
📊 Spark Job ID(s): [47, 46]

2️⃣ Images presque carrées (aspect ratio entre 0.9 et 1.1)

🔵 JOB 23: Images carrées
📌 Tracking ID: d2df9402
+--------------------+-----+------+------------+
|            filename|width|height|aspect_ratio|
+--------------------+-----+------+------------+
|  Abyssinian_178.jpg|  400|   400|        1.00|
|german_shorthaire...|  500|   500|        1.00|
|english_cocker_sp...|  500|   500|        1.00|
|  leonberger_104.jpg|  500|   460|        1.09|
|     keeshond_61.jpg|  500|   465|        1.08|
|newfoundland_180.jpg|  500|   500|        1.00|
|newfoundland_190.jpg|  500|   500|        1.00|
|newfoundland_129.jpg|  500|   500|        1.00|
|    shiba_inu_84.jpg|  5

In [ ]:
# ============================================
# PARTIE 14 : Comparaison Formats de Stockage
# ============================================

print("\n⚖️ PARTIE 14 : Parquet vs Fichiers Bruts")
print("="*70)

print("""
💡 COMPARAISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fichiers JPG bruts :
  ✓ Format image standard
  ✓ Compression optimale pour images
  ✗ Pas de métadonnées structurées
  ✗ Pas de partition pruning
  ✗ Lecture séquentielle uniquement

Parquet avec images binaires :
  ✓ Métadonnées + images dans un seul format
  ✓ Partitionnement (lecture sélective)
  ✓ Schéma structuré
  ✓ Compression columnar
  ✗ Légèrement plus gros (métadonnées)

Recommandation :
  • Images brutes : stockage long terme, partage
  • Parquet : pipelines Spark, analytics, features
""")

# Comparer les tailles
import subprocess

original_size = subprocess.check_output(['du', '-sh', images_path]).split()[0].decode('utf-8')
parquet_size = subprocess.check_output(['du', '-sh', output_path]).split()[0].decode('utf-8')

print(f"\n💾 Comparaison des tailles (1000 images) :")
print(f"   Fichiers JPG originaux : ~{original_size} (pour toutes les images)")
print(f"   Parquet avec métadonnées : {parquet_size}")


⚖️ PARTIE 14 : Parquet vs Fichiers Bruts

💡 COMPARAISON
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Fichiers JPG bruts :
  ✓ Format image standard
  ✓ Compression optimale pour images
  ✗ Pas de métadonnées structurées
  ✗ Pas de partition pruning
  ✗ Lecture séquentielle uniquement

Parquet avec images binaires :
  ✓ Métadonnées + images dans un seul format
  ✓ Partitionnement (lecture sélective)
  ✓ Schéma structuré
  ✓ Compression columnar
  ✗ Légèrement plus gros (métadonnées)

Recommandation :
  • Images brutes : stockage long terme, partage
  • Parquet : pipelines Spark, analytics, features


💾 Comparaison des tailles (1000 images) :
   Fichiers JPG originaux : ~775M (pour toutes les images)
   Parquet avec métadonnées : 192M


In [ ]:
# ============================================
# PARTIE 15 : Analyse Exploratoire Finale
# ============================================

print("\n📊 PARTIE 15 : Analyse Exploratoire Finale")
print("="*70)

print("\n1️⃣ Images les plus grandes et les plus petites")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.orderBy(F.col("pixels").desc())
        .select("filename", "race", "width", "height", "pixels")
        .show(5),
    "JOB 25: Plus grandes images"
)

print("\n2️⃣ Races avec images moyennes les plus grandes")

result = monitor.execute_and_monitor(
    lambda: df_with_dims.groupBy("race", "species")
        .agg(
            F.avg("pixels").alias("pixels_moyen"),
            F.count("*").alias("nb_images")
        )
        .orderBy(F.col("pixels_moyen").desc())
        .limit(10)
        .show(truncate=False),
    "JOB 26: Races avec plus grandes images"
)

print("\n3️⃣ Distribution de la luminosité par espèce")

# Créer le DataFrame avec features
df_with_features = df_sample.withColumn(
    "color_features",
    extract_color_features(F.col("content"))
).withColumn(
    "brightness",
    F.col("color_features.brightness")
).drop("color_features")

result = monitor.execute_and_monitor(
    lambda: df_with_features.groupBy("species")
        .agg(
            F.avg("brightness").alias("luminosite_moyenne"),
            F.min("brightness").alias("luminosite_min"),
            F.max("brightness").alias("luminosite_max")
        )
        .show(),
    "JOB 27: Luminosité par espèce"
)


📊 PARTIE 15 : Analyse Exploratoire Finale

1️⃣ Images les plus grandes et les plus petites

🔵 JOB 25: Plus grandes images
📌 Tracking ID: d10aece9
+--------------------+------------+-----+------+-------+
|            filename|        race|width|height| pixels|
+--------------------+------------+-----+------+-------+
|Egyptian_Mau_162.jpg|Egyptian Mau| 3264|  2448|7990272|
|Egyptian_Mau_165.jpg|Egyptian Mau| 2560|  1920|4915200|
|Egyptian_Mau_196.jpg|Egyptian Mau| 1886|  2606|4914916|
| Egyptian_Mau_20.jpg|Egyptian Mau| 1440|  2160|3110400|
|Egyptian_Mau_182.jpg|Egyptian Mau| 1999|  1499|2996501|
+--------------------+------------+-----+------+-------+
only showing top 5 rows

✅ SUCCESS
⏱️  Durée: 3.05s
📊 Spark Job ID(s): [52]

2️⃣ Races avec images moyennes les plus grandes

🔵 JOB 26: Races avec plus grandes images
📌 Tracking ID: 3abac3dd
+----------------+-------+------------------+---------+
|race            |species|pixels_moyen      |nb_images|
+----------------+-------+-----------